In [ ]:
!pip install -q transformers accelerate datasets Pillow tqdm

In [ ]:
!git clone https://github.com/sht037-lgtm/Q-Vtree.git
%cd Q-Vtree

fatal: destination path 'Q-Vtree' already exists and is not an empty directory.
/content/Q-Vtree/Q-Vtree


In [ ]:
!git clone https://huggingface.co/datasets/craigwu/vstar_bench

from datasets import load_dataset
ds = load_dataset("craigwu/vstar_bench")
print(ds)

fatal: destination path 'vstar_bench' already exists and is not an empty directory.


In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor
import torch

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model loaded:", model.__class__.__name__)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Model loaded: LlavaForConditionalGeneration


In [ ]:
!cd /content/Q-Vtree && git pull

import sys, importlib

# 清除缓存
if "evaluate_llava" in sys.modules:
    del sys.modules["evaluate_llava"]

sys.path.insert(0, "LLaVA")
from evaluate_llava import run_vstar_inference_llava, evaluate_vstar_predictions

# 确认函数签名正确
import inspect
print(inspect.signature(run_vstar_inference_llava))

Already up to date.
(model, processor, dataset, dataset_dir: str = 'vstar_bench', output_file: str = 'vstar_predictions_llava15.jsonl', max_samples: int | None = None, max_new_tokens: int = 16, run_name: str = 'llava15_7b')


In [ ]:

pred_file = run_vstar_inference_llava(
    model=model,
    processor=processor,
    dataset=ds["test"],
    dataset_dir="vstar_bench",
    output_file="vstar_predictions_llava15.jsonl",
    run_name="llava15_7b",
)

evaluate_vstar_predictions(pred_file)

[LLaVA-1.5] V-Star: 100%|██████████| 191/191 [00:32<00:00,  5.88it/s]

[INFO] Predictions saved to: vstar_predictions_llava15.jsonl

  Overall Accuracy : 42.93%  (82/191)
  direct_attributes             : 37.39%  (43/115)
  relative_position             : 51.32%  (39/76)



{'overall': 0.4293193717277487,
 'per_category': {'direct_attributes': {'correct': 43, 'total': 115},
  'relative_position': {'correct': 39, 'total': 76}}}